# Chapter 6: Inference for Categorical Data
**Haydar Kilic — Artificial Intelligence Engineering**

This notebook covers the following topics with hands-on examples:

1. **Inference for a Single Proportion** — Confidence interval and hypothesis test
2. **Difference of Two Proportions** — Comparative inference
3. **Chi-Square Goodness-of-Fit Test** — Observed vs expected distribution
4. **Chi-Square Test of Independence** — Relationship between two categorical variables

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats
from scipy.stats import norm, chi2
import warnings
warnings.filterwarnings('ignore')

# Plot settings
plt.rcParams['figure.figsize'] = (9, 5)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

print("Libraries loaded successfully ✓")

---
## 1. Inference for a Single Proportion

### 1.1 Theoretical Background

By the **Central Limit Theorem**, the sample proportion $\hat{p}$ for a population proportion $p$ is approximately normally distributed:

$$\hat{p} \sim N\left(p,\ SE = \sqrt{\frac{p(1-p)}{n}}\right)$$

**Conditions:**
- Independence: Random sample + 10% condition
- Success-failure: At least 10 successes and 10 failures

**95% Confidence Interval:**
$$\hat{p} \pm 1.96 \times \sqrt{\frac{\hat{p}(1-\hat{p})}{n}}$$

### 1.2 GSS Example — Experimental Design Question

A GSS survey found that 571 out of 670 Americans (85%) answered an experimental design question correctly. Let us compute a 95% confidence interval for the proportion of all Americans who would answer this question correctly.

In [ ]:
# ── Data ─────────────────────────────────────────────────────────────────────
n          = 670        # sample size
successes  = 571        # number of correct responses
p_hat      = successes / n
conf_level = 0.95
z_star     = norm.ppf(1 - (1 - conf_level) / 2)   # critical value

# ── Condition Check ───────────────────────────────────────────────────────────
print("=" * 50)
print("CONDITION CHECK")
print("=" * 50)
print(f"Successes  : {successes}  {'✓ (≥10)' if successes >= 10 else '✗ (<10)'}")
print(f"Failures   : {n - successes}  {'✓ (≥10)' if (n - successes) >= 10 else '✗ (<10)'}")
print(f"Sample proportion : {p_hat:.4f}")

# ── Standard Error ────────────────────────────────────────────────────────────
SE     = np.sqrt(p_hat * (1 - p_hat) / n)
margin = z_star * SE
CI_low = p_hat - margin
CI_high= p_hat + margin

print("\n" + "=" * 50)
print("CALCULATIONS")
print("=" * 50)
print(f"Critical value (z*)  : {z_star:.4f}")
print(f"Standard error (SE)  : {SE:.4f}")
print(f"Margin of error (ME) : ±{margin:.4f}")
print(f"\n95% Confidence Interval : ({CI_low:.4f}, {CI_high:.4f})")
print(f"                         = ({CI_low*100:.1f}%, {CI_high*100:.1f}%)")

In [ ]:
# ── Visualization: Confidence Interval ───────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4))

x = np.linspace(0.78, 0.92, 400)
y = norm.pdf(x, p_hat, SE)

# Distribution curve
ax.plot(x, y, 'steelblue', lw=2.5)

# Confidence interval shading
x_fill = np.linspace(CI_low, CI_high, 300)
ax.fill_between(x_fill, norm.pdf(x_fill, p_hat, SE),
                alpha=0.35, color='steelblue', label='95% CI')

# Boundary lines
for val, label in [(CI_low, f'{CI_low:.3f}'), (CI_high, f'{CI_high:.3f}')]:
    ax.axvline(val, color='navy', lw=1.5, ls='--')
    ax.text(val, ax.get_ylim()[1] * 0.02, label,
            ha='center', va='bottom', fontsize=10, color='navy')

ax.axvline(p_hat, color='crimson', lw=2, label=f'$\\hat{{p}}$ = {p_hat:.3f}')

ax.set_xlabel('Sample Proportion ($\\hat{p}$)', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('95% Confidence Interval — Experimental Design Question (n=670)', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

print(f"\nInterpretation: We are 95% confident that between {CI_low*100:.1f}% and {CI_high*100:.1f}%")
print("of all Americans would answer the experimental design question correctly.")

### 1.3 Hypothesis Test — Single Proportion

**Question:** Do these data provide convincing evidence that more than 80% of Americans have good intuition about experimental design?

$$H_0: p = 0.80 \qquad H_A: p > 0.80$$

**Note:** In a hypothesis test, the standard error is computed using the null value $p_0$.

In [ ]:
# ── Hypothesis Test ──────────────────────────────────────────────────────────
p_0   = 0.80    # null value
p_hat = 571 / 670
n     = 670

# Condition check (using H0 value)
exp_success = n * p_0
exp_failure = n * (1 - p_0)
print("CONDITION CHECK (using H0 value for HT):")
print(f"  Expected successes : {exp_success:.0f}  {'✓' if exp_success >= 10 else '✗'}")
print(f"  Expected failures  : {exp_failure:.0f}  {'✓' if exp_failure >= 10 else '✗'}")

# Test statistic
SE_ht = np.sqrt(p_0 * (1 - p_0) / n)
Z     = (p_hat - p_0) / SE_ht

# p-value (right-tail test)
p_value = 1 - norm.cdf(Z)

print("\nHYPOTHESIS TEST")
print(f"  H0 : p = {p_0}")
print(f"  HA : p > {p_0}")
print(f"  SE = sqrt({p_0}×{1-p_0}/{n}) = {SE_ht:.4f}")
print(f"  Z  = ({p_hat:.4f} - {p_0}) / {SE_ht:.4f} = {Z:.2f}")
print(f"  p-value = P(Z > {Z:.2f}) = {p_value:.4f}")
print(f"\n  Decision (α=0.05): {'REJECT H0 ✓' if p_value < 0.05 else 'Fail to reject H0'}")

In [ ]:
# ── Visualization: Z-test ─────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4))

x = np.linspace(-4, 5, 400)
y = norm.pdf(x)

ax.plot(x, y, 'steelblue', lw=2.5)

# p-value area (right tail)
x_fill = np.linspace(Z, 5, 200)
ax.fill_between(x_fill, norm.pdf(x_fill), alpha=0.45,
                color='crimson', label=f'p-value = {p_value:.4f}')

ax.axvline(Z, color='crimson', lw=2, ls='--', label=f'Z = {Z:.2f}')
ax.axvline(0, color='gray', lw=1, ls=':')

ax.set_xlabel('Z value', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('Hypothesis Test: $H_0: p=0.80$ vs $H_A: p>0.80$', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

print("Interpretation: Since p-value < 0.05, we reject H0.")
print("The data provide convincing evidence that more than 80% of Americans")
print("have good intuition about experimental design.")

### 1.4 Choosing Sample Size

The sample size required to achieve a given margin of error:

$$n \geq \frac{z^{*2} \cdot \hat{p}(1-\hat{p})}{ME^2}$$

**If no prior study exists:** Use $\hat{p} = 0.5$ (the most conservative estimate, giving the maximum $n$).

In [ ]:
# ── Sample Size Calculation ───────────────────────────────────────────────────
import math

def required_sample_size(ME, confidence=0.95, p_hat=None):
    """
    ME         : Desired margin of error (e.g. 0.01 = ±1%)
    confidence : Confidence level
    p_hat      : Prior estimate; if None, uses p_hat=0.5 (conservative)
    """
    z = norm.ppf(1 - (1 - confidence) / 2)
    if p_hat is None:
        p_hat   = 0.5
        note    = "(p̂=0.5 used — conservative estimate)"
    else:
        note    = f"(p̂={p_hat} used — from prior study)"
    n = (z**2 * p_hat * (1 - p_hat)) / ME**2
    return math.ceil(n), note

# Example 1: ME = 1%, prior p̂ = 0.85
n1, note1 = required_sample_size(ME=0.01, p_hat=0.85)
# Example 2: ME = 1%, no prior study
n2, note2 = required_sample_size(ME=0.01, p_hat=None)
# Example 3: ME = 3%
n3, note3 = required_sample_size(ME=0.03, p_hat=None)

print("SAMPLE SIZE CALCULATION (95% confidence level)")
print("-" * 60)
print(f"ME = ±1%, {note1}")
print(f"  → At least n = {n1} people must be sampled\n")
print(f"ME = ±1%, {note2}")
print(f"  → At least n = {n2} people must be sampled\n")
print(f"ME = ±3%, {note3}")
print(f"  → At least n = {n3} people must be sampled")

In [ ]:
# ── Required n by p̂ value — visualization ────────────────────────────────────
p_values = np.linspace(0.01, 0.99, 200)
z        = norm.ppf(0.975)   # z*=1.96 for 95%

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for i, ME in enumerate([0.01, 0.03]):
    n_vals = np.ceil((z**2 * p_values * (1 - p_values)) / ME**2)
    axes[i].plot(p_values, n_vals, 'steelblue', lw=2.5)
    axes[i].axvline(0.5, color='crimson', lw=1.5, ls='--', label='p̂=0.5 (max)')
    axes[i].fill_between(p_values, n_vals, alpha=0.15, color='steelblue')
    axes[i].set_xlabel('$\\hat{p}$ (Estimated proportion)', fontsize=11)
    axes[i].set_ylabel('Required n', fontsize=11)
    axes[i].set_title(f'Required Sample Size for ME = ±{int(ME*100)}%', fontsize=12)
    axes[i].legend(fontsize=10)

plt.tight_layout()
plt.show()

print("Note: p̂=0.5 yields the largest sample size → the most conservative estimate.")

---
## 2. Difference of Two Proportions

### 2.1 Theoretical Background

Standard error of the difference of proportions for two independent groups:

$$SE_{(\hat{p}_1 - \hat{p}_2)} = \sqrt{\frac{p_1(1-p_1)}{n_1} + \frac{p_2(1-p_2)}{n_2}}$$

**Confidence interval:** $(\hat{p}_1 - \hat{p}_2) \pm z^* \times SE$

**In the hypothesis test** ($H_0: p_1 = p_2$), the **pooled proportion** is used:

$$\hat{p}_{pool} = \frac{\text{successes}_1 + \text{successes}_2}{n_1 + n_2}$$

### 2.2 Melting Ice Sheets — Duke vs US

In [ ]:
# ── Data ─────────────────────────────────────────────────────────────────────
duke_yes   = 69;  duke_total = 105
us_yes     = 454; us_total   = 680

p_duke = duke_yes   / duke_total
p_us   = us_yes     / us_total
diff   = p_duke - p_us

# Summary table
df_data = pd.DataFrame({
    'Group':              ['Duke Students', 'US General Public'],
    'Very Concerned':     [duke_yes, us_yes],
    'Total (n)':          [duke_total, us_total],
    'Proportion (p̂)':    [f'{p_duke:.3f}', f'{p_us:.3f}']
})
print(df_data.to_string(index=False))
print(f"\nDifference in proportions: p̂_Duke - p̂_US = {p_duke:.3f} - {p_us:.3f} = {diff:.3f}")

In [ ]:
# ── 95% Confidence Interval ───────────────────────────────────────────────────
z_star = norm.ppf(0.975)

SE_diff = np.sqrt(
    p_duke * (1 - p_duke) / duke_total +
    p_us   * (1 - p_us)   / us_total
)

CI_low  = diff - z_star * SE_diff
CI_high = diff + z_star * SE_diff

print("TWO-PROPORTION DIFFERENCE — CONFIDENCE INTERVAL")
print(f"  SE = sqrt({p_duke:.3f}×{1-p_duke:.3f}/{duke_total} + {p_us:.3f}×{1-p_us:.3f}/{us_total})")
print(f"     = {SE_diff:.4f}")
print(f"  ME = {z_star:.2f} × {SE_diff:.4f} = ±{z_star*SE_diff:.4f}")
print(f"\n  95% CI = ({CI_low:.3f}, {CI_high:.3f})")
print(f"\n  Does the CI contain 0? {'YES' if CI_low < 0 < CI_high else 'NO'}")
if CI_low < 0 < CI_high:
    print("  → There is NO statistically significant difference between the two proportions.")

In [ ]:
# ── Hypothesis Test — Pooled Proportion ───────────────────────────────────────
p_pool = (duke_yes + us_yes) / (duke_total + us_total)

SE_pool = np.sqrt(
    p_pool * (1 - p_pool) / duke_total +
    p_pool * (1 - p_pool) / us_total
)

Z_ht    = diff / SE_pool
p_value = 2 * (1 - norm.cdf(abs(Z_ht)))   # two-tailed

print("HYPOTHESIS TEST — POOLED PROPORTION")
print(f"  H0 : p_Duke = p_US")
print(f"  HA : p_Duke ≠ p_US")
print(f"\n  Pooled proportion:")
print(f"  p̂_pool = ({duke_yes}+{us_yes}) / ({duke_total}+{us_total}) = {p_pool:.3f}")
print(f"  SE      = {SE_pool:.4f}")
print(f"  Z       = {diff:.3f} / {SE_pool:.4f} = {Z_ht:.2f}")
print(f"  p-value = 2×P(Z < {Z_ht:.2f}) = {p_value:.4f}")
print(f"\n  Decision (α=0.05): {'REJECT H0' if p_value < 0.05 else 'Fail to reject H0 ✓'}")
print("  Interpretation: No statistically significant difference in proportions was found between the two groups.")

In [ ]:
# ── CI vs HT Standard Error Comparison ───────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4))

x = np.linspace(-0.15, 0.15, 400)

# CI distribution
y_ci = norm.pdf(x, diff, SE_diff)
ax.plot(x, y_ci, 'steelblue', lw=2.5, label=f'CI distribution (SE={SE_diff:.4f})')

# CI shading
x_fill = np.linspace(CI_low, CI_high, 300)
ax.fill_between(x_fill, norm.pdf(x_fill, diff, SE_diff),
                alpha=0.25, color='steelblue', label='95% CI')

# Zero line
ax.axvline(0,    color='crimson', lw=2,  ls='--', label='0 (no difference)')
ax.axvline(diff, color='navy',    lw=2,  label=f'Observed difference = {diff:.3f}')

# CI boundaries
for val in [CI_low, CI_high]:
    ax.axvline(val, color='steelblue', lw=1, ls=':')
    ax.text(val, -0.3, f'{val:.3f}', ha='center', fontsize=9, color='steelblue')

ax.set_xlabel('$\\hat{p}_{Duke} - \\hat{p}_{US}$', fontsize=12)
ax.set_title('Duke vs US — 95% Confidence Interval for Difference of Two Proportions', fontsize=12, fontweight='bold')
ax.legend(fontsize=9)
ax.set_ylim(bottom=-0.5)
plt.tight_layout()
plt.show()

### 2.3 Reference: Difference Between CI and HT Standard Errors

| | Confidence Interval (CI) | Hypothesis Test (HT) |
|---|---|---|
| **Single proportion** | $SE = \sqrt{\frac{\hat{p}(1-\hat{p})}{n}}$ | $SE = \sqrt{\frac{p_0(1-p_0)}{n}}$ |
| **Two proportions** | $SE = \sqrt{\frac{\hat{p}_1(1-\hat{p}_1)}{n_1} + \frac{\hat{p}_2(1-\hat{p}_2)}{n_2}}$ | $SE = \sqrt{\hat{p}_{pool}(1-\hat{p}_{pool})\left(\frac{1}{n_1}+\frac{1}{n_2}\right)}$ |

---
## 3. Chi-Square Goodness-of-Fit Test

### 3.1 Theoretical Background

The **chi-square (χ²)** statistic is used to measure how well observed counts fit an expected distribution:

$$\chi^2 = \sum_{i=1}^{k} \frac{(O_i - E_i)^2}{E_i}$$

- $O_i$: Observed count
- $E_i$: Expected count  
- $k$: Number of categories
- **Degrees of freedom:** $df = k - 1$
- **p-value:** Right tail of the χ² curve

### 3.2 Chi-Square Distribution — Visualization

In [ ]:
# ── Chi-Square Distribution — Different df Values ─────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))

x      = np.linspace(0.01, 30, 500)
colors = ['#1f77b4', '#2ca02c', '#d62728']
styles = ['-', '--', ':']

for df, color, style in zip([2, 4, 9], colors, styles):
    y = chi2.pdf(x, df=df)
    ax.plot(x, y, color=color, lw=2.5, ls=style, label=f'df = {df}')

ax.set_xlim(0, 28)
ax.set_ylim(0, 0.45)
ax.set_xlabel('χ²', fontsize=13)
ax.set_ylabel('Density', fontsize=13)
ax.set_title('Chi-Square Distribution — Different Degrees of Freedom', fontsize=14, fontweight='bold')
ax.legend(fontsize=12, title='Degrees of Freedom')

# Annotation
ax.annotate('As df increases:\n• Centre shifts right\n• Variability increases\n• Approaches normal',
            xy=(18, 0.25), fontsize=10,
            bbox=dict(boxstyle='round,pad=0.4', facecolor='lightyellow', alpha=0.8))

plt.tight_layout()
plt.show()

# Convergence to normal as df → ∞
print("NOTE: As df increases, the χ² distribution approaches the normal.")
print("      Mean of χ² = df,  Variance of χ² = 2×df")

### 3.3 Labby's Dice

In [ ]:
# ── Labby Dice Experiment Data ────────────────────────────────────────────────
outcomes    = [1, 2, 3, 4, 5, 6]
observed    = np.array([53222, 52118, 52465, 52338, 52244, 53285])
total_rolls = observed.sum()

# Expected under equal probability assumption
expected    = np.array([total_rolls / 6] * 6)

# Chi-square contributions
contributions = (observed - expected)**2 / expected
chi2_stat     = contributions.sum()
df            = len(outcomes) - 1
p_value_dice  = chi2.sf(chi2_stat, df=df)   # sf = 1 - cdf = right tail

# Summary table
df_dice = pd.DataFrame({
    'Outcome':            outcomes,
    'Observed (O)':       observed,
    'Expected (E)':       expected.astype(int),
    'O - E':              (observed - expected).astype(int),
    '(O-E)²/E':           contributions.round(2)
})

print("LABBY'S DICE — Goodness-of-Fit Test")
print(df_dice.to_string(index=False))
print("-" * 55)
print(f"χ² = {chi2_stat:.2f}   df = {df}   p-value = {p_value_dice:.6f}")
print(f"\nDecision (α=0.05): {'REJECT H0 — Dice are biased!' if p_value_dice < 0.05 else 'Fail to reject H0 — Dice appear fair'}")

In [ ]:
# ── Visualization: Chi-Square p-value ────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Left: Chi-square distribution and p-value
ax = axes[0]
x  = np.linspace(0, 35, 500)
y  = chi2.pdf(x, df=df)
ax.plot(x, y, 'steelblue', lw=2.5)

x_fill = np.linspace(chi2_stat, 35, 300)
ax.fill_between(x_fill, chi2.pdf(x_fill, df=df),
                alpha=0.5, color='crimson', label=f'p-value = {p_value_dice:.5f}')
ax.axvline(chi2_stat, color='crimson', lw=2, ls='--', label=f'χ² = {chi2_stat:.2f}')
ax.set_xlabel('χ²', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title(f'Chi-Square Distribution (df={df})', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)

# Right: Observed vs Expected bar chart
ax2   = axes[1]
x_pos = np.arange(len(outcomes))
width = 0.35
ax2.bar(x_pos - width/2, observed, width,
        color='steelblue', label='Observed', alpha=0.85)
ax2.bar(x_pos + width/2, expected, width,
        color='gray',      label='Expected', alpha=0.85)
ax2.axhline(expected[0], color='gray', lw=1.5, ls='--', alpha=0.7)
ax2.set_xticks(x_pos)
ax2.set_xticklabels([f'Face {s}' for s in outcomes])
ax2.set_ylabel('Count', fontsize=12)
ax2.set_title("Observed vs Expected Counts", fontsize=12, fontweight='bold')
ax2.legend(fontsize=10)

plt.tight_layout()
plt.show()

### 3.4 2009 Iranian Election Example

In [ ]:
# ── 2009 Iranian Election ─────────────────────────────────────────────────────
candidates         = ['Ahmadinejad', 'Mousavi', 'Minor Candidates']
observed_iran      = np.array([338, 136, 30])    # votes in the survey
reported_proportion = np.array([0.6329, 0.3410, 0.0261])  # official election results
n_iran             = observed_iran.sum()

expected_iran      = n_iran * reported_proportion
contribution_iran  = (observed_iran - expected_iran)**2 / expected_iran
chi2_iran          = contribution_iran.sum()
df_iran_val        = len(candidates) - 1
p_iran             = chi2.sf(chi2_iran, df=df_iran_val)

df_iran = pd.DataFrame({
    'Candidate':         candidates,
    'Observed (O)':      observed_iran,
    'Reported %':        (reported_proportion * 100).round(2),
    'Expected (E)':      expected_iran.round(1),
    '(O-E)²/E':          contribution_iran.round(2)
})

print("2009 IRANIAN ELECTION — Goodness-of-Fit Test")
print(df_iran.to_string(index=False))
print("-" * 65)
print(f"χ² = {chi2_iran:.2f}   df = {df_iran_val}   p-value = {p_iran:.6f}")
print(f"\nDecision (α=0.05): {'REJECT H0' if p_iran < 0.05 else 'Fail to reject H0'}")
print("Interpretation: The survey data are inconsistent with the reported")
print("       election results — pointing to possible election fraud.")

---
## 4. Chi-Square Test of Independence

### 4.1 Theoretical Background

Tests independence between two categorical variables.

**Hypotheses:**
- $H_0$: The two variables are independent
- $H_A$: The two variables are dependent

**Expected count:** $E = \dfrac{\text{row total} \times \text{column total}}{\text{table total}}$

**Degrees of freedom:** $df = (\text{rows} - 1) \times (\text{columns} - 1)$

### 4.2 Popular Kids Example

In [ ]:
# ── Data ─────────────────────────────────────────────────────────────────────
observed_mat = np.array([
    [63, 31, 25],   # 4th grade
    [88, 55, 33],   # 5th grade
    [96, 55, 32]    # 6th grade
])

row_labels = ['4th grade', '5th grade', '6th grade']
col_labels = ['Grades', 'Popularity', 'Sports']

df_pop = pd.DataFrame(observed_mat, index=row_labels, columns=col_labels)
df_pop['Row Total'] = df_pop.sum(axis=1)

print("OBSERVED COUNTS:")
print(df_pop)
print(f"\nColumn Totals: {observed_mat.sum(axis=0)}")
print(f"Grand Total  : {observed_mat.sum()}")

In [ ]:
# ── Compute Expected Counts ───────────────────────────────────────────────────
grand_total = observed_mat.sum()
row_totals  = observed_mat.sum(axis=1)
col_totals  = observed_mat.sum(axis=0)

# E = row_total × col_total / grand_total
expected_mat = np.outer(row_totals, col_totals) / grand_total

df_expected = pd.DataFrame(expected_mat.round(1), index=row_labels, columns=col_labels)
print("EXPECTED COUNTS (E = row_total × col_total / grand_total):")
print(df_expected)

# Condition check: all cells ≥ 5?
print(f"\nCONDITION: All expected counts ≥ 5? {'✓ YES' if (expected_mat >= 5).all() else '✗ NO'}")

In [ ]:
# ── Chi-Square Statistic ──────────────────────────────────────────────────────
chi2_pop, p_pop, df_pop_val, expected_scipy = stats.chi2_contingency(
    observed_mat, correction=False
)

# Manual calculation
contributions_mat = (observed_mat - expected_mat)**2 / expected_mat
chi2_manual       = contributions_mat.sum()

print("CHI-SQUARE TEST STATISTIC")
print("\n(O-E)²/E cell contributions:")
df_contrib = pd.DataFrame(contributions_mat.round(4), index=row_labels, columns=col_labels)
print(df_contrib)
print("-" * 40)
print(f"χ² = {chi2_manual:.4f}")
print(f"df = ({len(row_labels)}-1) × ({len(col_labels)}-1) = {df_pop_val}")
print(f"p-value = {p_pop:.4f}")
print(f"\nDecision (α=0.05): {'REJECT H0' if p_pop < 0.05 else 'Fail to reject H0 — Grade and goals are INDEPENDENT ✓'}")

In [ ]:
# ── Visualization ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Stacked bar chart
ax     = axes[0]
x_pos  = np.arange(len(row_labels))
width  = 0.5
colors = ['#4e79a7', '#f28e2b', '#59a14f']

bottom = np.zeros(len(row_labels))
for i, (col, color) in enumerate(zip(col_labels, colors)):
    values = observed_mat[:, i]
    ax.bar(x_pos, values, width, bottom=bottom,
           color=color, label=col, alpha=0.9)
    bottom += values

ax.set_xticks(x_pos)
ax.set_xticklabels(row_labels)
ax.set_ylabel('Number of Students', fontsize=12)
ax.set_title('Goal Distribution by Grade', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)

# Right: Normalised (percentage) stacked bar chart
ax2          = axes[1]
row_sums     = observed_mat.sum(axis=1, keepdims=True)
pct          = observed_mat / row_sums * 100

bottom = np.zeros(len(row_labels))
for i, (col, color) in enumerate(zip(col_labels, colors)):
    values = pct[:, i]
    bars = ax2.bar(x_pos, values, width, bottom=bottom,
                   color=color, label=col, alpha=0.9)
    # Percentage labels
    for bar, val, bot in zip(bars, values, bottom):
        if val > 4:
            ax2.text(bar.get_x() + bar.get_width()/2,
                     bot + val/2, f'{val:.0f}%',
                     ha='center', va='center', fontsize=9, color='white', fontweight='bold')
    bottom += values

ax2.set_xticks(x_pos)
ax2.set_xticklabels(row_labels)
ax2.set_ylabel('Percentage (%)', fontsize=12)
ax2.set_title('Normalised Goal Distribution by Grade', fontsize=12, fontweight='bold')
ax2.legend(fontsize=10)
ax2.set_ylim(0, 110)

plt.tight_layout()
plt.show()

print("Observation: Percentage distributions are very similar across grades → independence")

### 4.3 Helper Function: General Chi-Square Test

In [ ]:
# ── General-Purpose Chi-Square Test Function ──────────────────────────────────
def chi_square_independence_test(obs_mat, row_names, col_names, alpha=0.05):
    """
    Test of independence between two categorical variables.
    Takes an observed matrix plus row and column names.
    """
    obs          = np.array(obs_mat)
    grand_total  = obs.sum()
    row_totals   = obs.sum(axis=1)
    col_totals   = obs.sum(axis=0)
    expected     = np.outer(row_totals, col_totals) / grand_total

    chi2_val = ((obs - expected)**2 / expected).sum()
    df_val   = (obs.shape[0] - 1) * (obs.shape[1] - 1)
    p_val    = chi2.sf(chi2_val, df=df_val)

    # Result
    print(f"χ² = {chi2_val:.4f},  df = {df_val},  p-value = {p_val:.4f}")
    if p_val < alpha:
        print(f"→ REJECT H0 (p={p_val:.4f} < α={alpha}): Variables are DEPENDENT.")
    else:
        print(f"→ Fail to reject H0 (p={p_val:.4f} ≥ α={alpha}): Variables are INDEPENDENT.")
    return chi2_val, df_val, p_val, expected


# ── Additional Example: Gender × Pet Preference ───────────────────────────────
print("ADDITIONAL EXAMPLE: Is there an association between")
print("preferred pet and gender in a survey?\n")

# Observed: cat, dog, fish
#           male, female
sample_data = [
    [22, 48, 10],  # Male
    [38, 32, 20],  # Female
]

print("Observed:")
df_ex = pd.DataFrame(sample_data, index=['Male', 'Female'], columns=['Cat', 'Dog', 'Fish'])
df_ex['Total'] = df_ex.sum(axis=1)
print(df_ex, "\n")

chi_square_independence_test(
    sample_data,
    row_names=['Male', 'Female'],
    col_names=['Cat', 'Dog', 'Fish']
)

---
## 5. Summary: Comparison of All Methods

In [ ]:
# ── Comprehensive Summary Table ───────────────────────────────────────────────
summary = pd.DataFrame({
    'Method': [
        'Single Proportion CI',
        'Single Proportion HT',
        'Two Proportions Diff CI',
        'Two Proportions Diff HT',
        'Goodness-of-Fit χ²',
        'Independence χ²'
    ],
    'Parameter':      ['p', 'p', 'p₁-p₂', 'p₁-p₂', 'Distribution', 'Independence'],
    'Standard Error': [
        'sqrt(p̂(1-p̂)/n)',
        'sqrt(p₀(1-p₀)/n)',
        'sqrt(p̂₁(1-p̂₁)/n₁ + p̂₂(1-p̂₂)/n₂)',
        'sqrt(p̂pool(1-p̂pool)(1/n₁+1/n₂))',
        'Σ(O-E)²/E',
        'Σ(O-E)²/E'
    ],
    'Distribution': ['Z~N(0,1)', 'Z~N(0,1)', 'Z~N(0,1)', 'Z~N(0,1)', 'χ²(k-1)', 'χ²((r-1)(c-1))'],
    'Condition': [
        '≥10 obs successes/failures',
        '≥10 exp successes/failures',
        '≥10 for each group',
        '≥10 for each group',
        'Each cell E≥5, df>1',
        'Each cell E≥5, df>1'
    ]
})

print("INFERENCE FOR CATEGORICAL DATA — SUMMARY")
print("=" * 90)
print(summary.to_string(index=False))

In [ ]:
# ── Which Test Should I Use? — Decision Flowchart ─────────────────────────────
print("""
WHICH TEST SHOULD I USE?
═════════════════════════

         Categorical Data
               │
    ┌──────────┴──────────┐
    │                     │
  One group           Two groups
    │                     │
    │            ┌────────┴────────┐
    │          For CI          For HT
    │        Two-Prop         Two-Prop
    │        Diff CI          Diff HT
    │       (use p̂₁,p̂₂      (use p̂_pool)
    │        directly)
    │
  ┌─┴──────────────┐
  │                │
CI/HT       Multiple categories
(Z test)           │
               ┌────┴────┐
           One var.   Two var.
          Goodness-  Independence
          of-Fit     Chi-Square
          Chi-Square (df=(r-1)(c-1))
          (df=k-1)
""")

---
## 6. Exercises

**Exercise 1:** A Gallup poll found that 11% of 1001 Americans object to Halloween for religious reasons. If the 95% confidence interval has a margin of error of ±3%, does this support the claim that more than 10% of all Americans object?

**Exercise 2:** An A/B test is conducted on two different websites. On Site A, 32 out of 200 visitors made a purchase; on Site B, 45 out of 180 visitors made a purchase. Is there a significant difference in conversion rates between the two sites?

**Exercise 3:** You rolled a die 120 times with these results: {1:18, 2:22, 3:17, 4:19, 5:25, 6:19}. Is the die fair? (5% significance level)

In [ ]:
# ── Exercise 1: Gallup — Confidence Interval Interpretation ──────────────────
n_gallup      = 1001
p_hat_gallup  = 0.11
z_star_95     = norm.ppf(0.975)
SE_g          = np.sqrt(p_hat_gallup * (1 - p_hat_gallup) / n_gallup)
ME_g          = z_star_95 * SE_g
CI_low_g      = p_hat_gallup - ME_g
CI_high_g     = p_hat_gallup + ME_g

print("EXERCISE 1 SOLUTION")
print(f"p̂ = {p_hat_gallup}, n = {n_gallup}")
print(f"ME  = {z_star_95:.2f} × {SE_g:.4f} = ±{ME_g:.4f} ≈ ±{ME_g*100:.1f}%")
print(f"95% CI = ({CI_low_g*100:.1f}%, {CI_high_g*100:.1f}%)")
print(f"\nClaim: More than 10% object")
print(f"Lower bound {'> 10% → CLAIM SUPPORTED' if CI_low_g > 0.10 else '< 10% → CLAIM NOT SUPPORTED ✗'}")
print(f"\nWhy? CI = ({CI_low_g*100:.1f}%, {CI_high_g*100:.1f}%) — includes 10%!")

In [ ]:
# ── Exercise 2: A/B Test ──────────────────────────────────────────────────────
n_A, success_A = 200, 32
n_B, success_B = 180, 45
p_A    = success_A / n_A
p_B    = success_B / n_B
p_pool = (success_A + success_B) / (n_A + n_B)

SE_ab = np.sqrt(p_pool * (1 - p_pool) * (1/n_A + 1/n_B))
Z_ab  = (p_A - p_B) / SE_ab
p_ab  = 2 * (1 - norm.cdf(abs(Z_ab)))

print("EXERCISE 2 SOLUTION — A/B Test")
print(f"p̂_A = {success_A}/{n_A} = {p_A:.3f}")
print(f"p̂_B = {success_B}/{n_B} = {p_B:.3f}")
print(f"p̂_pool = {p_pool:.3f}")
print(f"Z = {Z_ab:.3f},  p-value = {p_ab:.4f}")
print(f"Decision: {'Significant difference EXISTS' if p_ab < 0.05 else 'No significant difference'} (α=0.05)")

In [ ]:
# ── Exercise 3: Is the Die Fair? ─────────────────────────────────────────────
die_results = np.array([18, 22, 17, 19, 25, 19])
n_die       = die_results.sum()
exp_die     = np.array([n_die / 6] * 6)
chi2_die    = ((die_results - exp_die)**2 / exp_die).sum()
df_die      = len(die_results) - 1
p_die       = chi2.sf(chi2_die, df=df_die)

print("EXERCISE 3 SOLUTION — Fair Die Test")
print(f"Observed : {die_results}  (total={n_die})")
print(f"Expected : {exp_die}")
print(f"χ² = {chi2_die:.3f},  df = {df_die},  p-value = {p_die:.4f}")
print(f"Decision: Die {'is BIASED ✗' if p_die < 0.05 else 'appears fair ✓'} (α=0.05)")

---

## Notebook Summary

In this notebook we covered:

| Topic | Key Formula | Distribution |
|------|---------------|----------|
| Single proportion CI | $\hat{p} \pm z^* \sqrt{\hat{p}(1-\hat{p})/n}$ | Normal |
| Single proportion HT | $Z = (\hat{p}-p_0)/\sqrt{p_0(1-p_0)/n}$ | Normal |
| Two proportions diff CI | $(\hat{p}_1-\hat{p}_2) \pm z^* \cdot SE$ | Normal |
| Two proportions diff HT | $Z = (\hat{p}_1-\hat{p}_2)/SE_{pool}$ | Normal |
| Goodness-of-fit χ² | $\chi^2 = \sum(O-E)^2/E$, $df=k-1$ | Chi-Square |
| Independence χ² | $\chi^2 = \sum(O-E)^2/E$, $df=(r-1)(c-1)$ | Chi-Square |

**Key reminders:**
- For CI use $\hat{p}$; for HT use $p_0$
- For two-proportion HT, use the pooled proportion $\hat{p}_{pool}$
- The chi-square test is always a **right-tail** test
- If conditions are not met → use **randomisation** methods (Section 6.4)